## Google Colab Setup

This notebook can run either locally from the repo or on Google Colab.

Colab workflow:
- Clone the GitHub repo into `/content/XAllergen2.0`.
- Download a fresh ESM-2 snapshot from Hugging Face.
- Use the tracked baseline checkpoint and split RSA `.json.gz` artifacts directly from the cloned repo.

Required repo artifacts for Colab:
- `models/baseline_frozen_esm2.pt`
- `data/rsa/deepalgpro_train_rsa.json.gz`
- `data/rsa/deepalgpro_test_rsa.json.gz`


In [1]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
os.environ['XALLERGEN_RUN_TARGET'] = 'colab' if IS_COLAB else 'local'

if IS_COLAB:
    RUNTIME_ROOT = Path('/content/XAllergen2.0')
else:
    for _candidate in [Path.cwd(), *Path.cwd().parents]:
        if (_candidate / 'src' / 'xallergen').exists():
            RUNTIME_ROOT = _candidate
            break
    else:
        raise FileNotFoundError('Could not locate repo root with src/xallergen')

if IS_COLAB:
    if RUNTIME_ROOT.exists():
        shutil.rmtree(RUNTIME_ROOT)
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/Jeffateth/XAllergen2.0.git', str(RUNTIME_ROOT)],
        check=True,
    )

SRC_DIR = RUNTIME_ROOT / 'src'
PACKAGE_DIR = SRC_DIR / 'xallergen'
DATA_DIR = RUNTIME_ROOT / 'data'
RSA_DIR = DATA_DIR / 'rsa'
MODELS_DIR = RUNTIME_ROOT / 'models'
RESULTS_DIR = RUNTIME_ROOT / 'results'
HF_DIR = RUNTIME_ROOT / 'hf_models' / 'facebook_esm2_t6_8M_UR50D'
for path in [SRC_DIR, PACKAGE_DIR, DATA_DIR, RSA_DIR, MODELS_DIR, RESULTS_DIR, HF_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    from huggingface_hub import snapshot_download

    fresh_model_path = snapshot_download(
        repo_id='facebook/esm2_t6_8M_UR50D',
        local_dir=HF_DIR,
        local_dir_use_symlinks=False,
        force_download=True,
        resume_download=False,
    )
    os.environ['XALLERGEN_HF_MODEL_DIR'] = str(fresh_model_path)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'RUN_TARGET: {os.environ["XALLERGEN_RUN_TARGET"]}')
print(f'Runtime root: {RUNTIME_ROOT}')
print(f'Data dir: {DATA_DIR}')
print(f'RSA dir: {RSA_DIR}')
print(f'Models dir: {MODELS_DIR}')
print(f'Source package dir: {PACKAGE_DIR}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive directories ready:
  /content/drive/MyDrive/XAllergen2.0/models
  /content/drive/MyDrive/XAllergen2.0/results


HTTPError: HTTP Error 404: Not Found

# Baseline ESM-2 with SASA Regularization

This notebook extends the frozen ESM-2 baseline with an optional SASA regularization term driven by NetSurfP-3.0 RSA predictions.

Design notes:
- The original `03_baseline_model_esm2.ipynb` remains the baseline reference.
- `lambda_sasa=0.0` keeps the SASA branch inactive so the original classifier path is preserved.
- The baseline notebook tokenizes with `add_special_tokens=False`, so this notebook keeps that default to preserve comparability.
- The NetSurfP CSVs are residue-level tables keyed by sequence ID, and this notebook reduces them to fixed RSA vectors per protein before training.


In [ ]:
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split

from xallergen.baseline_notebook_utils import (
    RANDOM_STATE,
    build_tokenizer,
    load_baseline_checkpoint,
    seed_everything,
)
from xallergen.baseline_sasa_experiment import (
    SASAExperimentConfig,
    build_dataloader,
    compute_metrics,
    inspect_rsa_inputs,
    load_rsa_lookup_dicts,
    predict,
    run_lambda_sasa_sweep,
)

TRAIN_CSV = DATA_DIR / 'deepalgpro_train_cleaned.csv'
TEST_CSV = DATA_DIR / 'deepalgpro_test_cleaned.csv'
TRAIN_RSA_PATH = RSA_DIR / 'deepalgpro_train_rsa.json.gz'
TEST_RSA_PATH = RSA_DIR / 'deepalgpro_test_rsa.json.gz'
POSITIVES_SPLITB_CSV = DATA_DIR / 'positives_splitB.csv'
BASELINE_CHECKPOINT_PATH = MODELS_DIR / 'baseline_frozen_esm2.pt'
RUN_MODELS_DIR = MODELS_DIR
RUN_RESULTS_DIR = RESULTS_DIR
RUN_MODELS_DIR.mkdir(parents=True, exist_ok=True)
RUN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SASA_SUMMARY_PATH = RUN_RESULTS_DIR / 'classification' / 'baseline_sasa_sweep_summary.csv'

for required_path in [
    TRAIN_CSV,
    TEST_CSV,
    TRAIN_RSA_PATH,
    TEST_RSA_PATH,
    POSITIVES_SPLITB_CSV,
    BASELINE_CHECKPOINT_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required file: {required_path}')

CONFIG = SASAExperimentConfig(
    lambda_cls=1.0,
    lambda_values=(0.0, 0.1, 0.5, 1.0, 5.0),
    add_special_tokens=False,
)

seed_everything(RANDOM_STATE)
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Tokenizer adds special tokens: {CONFIG.add_special_tokens}')
print(f'Baseline checkpoint: {BASELINE_CHECKPOINT_PATH}')
print(f'Run models dir: {RUN_MODELS_DIR}')
print(f'Run results dir: {RUN_RESULTS_DIR}')


In [ ]:
train_df = pd.read_csv(TRAIN_CSV).copy()
test_df = pd.read_csv(TEST_CSV).copy()
for df in [train_df, test_df]:
    df['sequence_id'] = df['sequence_id'].astype(str).str.strip()
    df['sequence'] = df['sequence'].astype(str).str.strip().str.upper()
    df['label'] = df['label'].astype(int)

rsa_report_df = inspect_rsa_inputs(
    train_rsa_path=TRAIN_RSA_PATH,
    test_rsa_path=TEST_RSA_PATH,
    train_frame=train_df,
    test_frame=test_df,
)
display(rsa_report_df)

print(f'Train RSA file: {TRAIN_RSA_PATH}')
print(f'Test RSA file : {TEST_RSA_PATH}')


In [ ]:
train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=RANDOM_STATE,
    stratify=train_df['label'],
)
train_split_df = train_split_df.reset_index(drop=True).copy()
val_split_df = val_split_df.reset_index(drop=True).copy()

print(f'Train sequences: {len(train_split_df):,}')
print(f'Validation sequences: {len(val_split_df):,}')
print(f'Held-out test sequences: {len(test_df):,}')


In [ ]:
train_rsa_lookup, test_rsa_lookup, rsa_summary = load_rsa_lookup_dicts(
    train_rsa_path=TRAIN_RSA_PATH,
    test_rsa_path=TEST_RSA_PATH,
    train_frame=train_df,
    test_frame=test_df,
    add_special_tokens=CONFIG.add_special_tokens,
)
display(pd.DataFrame([rsa_summary['train'], rsa_summary['test']]))


In [ ]:
tokenizer = build_tokenizer()
val_loader_reference = build_dataloader(
    frame=val_split_df,
    rsa_lookup=train_rsa_lookup,
    tokenizer=tokenizer,
    batch_size=CONFIG.batch_size,
    shuffle=False,
    add_special_tokens=CONFIG.add_special_tokens,
)

baseline_model, _ = load_baseline_checkpoint(BASELINE_CHECKPOINT_PATH, DEVICE)
baseline_val_summary, baseline_val_predictions_df = predict(
    baseline_model,
    val_loader_reference,
    DEVICE,
    criterion=None,
    lambda_cls=CONFIG.lambda_cls,
    lambda_sasa=0.0,
    threshold=CONFIG.threshold,
)
baseline_val_metrics = compute_metrics(baseline_val_predictions_df, threshold=CONFIG.threshold)
print('Reference baseline checkpoint validation AUROC:', round(baseline_val_metrics['auroc'], 3))


In [ ]:
sasa_summary_df = run_lambda_sasa_sweep(
    config=CONFIG,
    train_split_df=train_split_df,
    val_split_df=val_split_df,
    test_df=test_df,
    train_rsa_lookup=train_rsa_lookup,
    test_rsa_lookup=test_rsa_lookup,
    positives_splitb_csv=POSITIVES_SPLITB_CSV,
    model_dir=RUN_MODELS_DIR,
    results_dir=RUN_RESULTS_DIR,
    device=DEVICE,
)
SASA_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
sasa_summary_df.to_csv(SASA_SUMMARY_PATH, index=False)
display(sasa_summary_df)



In [ ]:
lambda_zero_row = sasa_summary_df.loc[sasa_summary_df['lambda_sasa'].eq(0.0)].copy()
if not lambda_zero_row.empty:
    sweep_val_auroc = float(lambda_zero_row.iloc[0]['val_auroc'])
    print('Reference baseline checkpoint val AUROC:', round(baseline_val_metrics['auroc'], 3))
    print('lambda_sasa=0 sweep val AUROC       :', round(sweep_val_auroc, 3))
    print('Match to three decimals            :', round(baseline_val_metrics['auroc'], 3) == round(sweep_val_auroc, 3))
print(f'Saved SASA sweep summary to: {SASA_SUMMARY_PATH}')
